# Homework 3

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Load and Inspect the AReM Dataset

In [1]:
from pathlib import Path
import sys

print("Python:", sys.executable)
print("Working directory:", Path.cwd())

Python: C:\Users\19098\DSCI-552\.venv\Scripts\python.exe
Working directory: C:\Users\19098\DSCI-552\Homework3


In [2]:
from pathlib import Path

data_dir = Path("data") / "AReM"
csv_files = sorted(data_dir.glob("*/*.csv"))

print("Data folder exists:", data_dir.exists())
print("Number of CSV files:", len(csv_files))

for folder in sorted(data_dir.iterdir()):
    if folder.is_dir():
        print(f"{folder.name}: {len(list(folder.glob('*.csv')))} files")

Data folder exists: True
Number of CSV files: 88
bending1: 7 files
bending2: 6 files
cycling: 15 files
lying: 15 files
sitting: 15 files
standing: 15 files
walking: 15 files


In [3]:
import pandas as pd

columns = [
    "time",
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"
]


def read_instance(path):
    rows = []

    for line_number, line in enumerate(
        path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        line = line.rstrip(", \t")

        fields = line.split(",") if "," in line else line.split()

        if len(fields) != 7:
            raise ValueError(
                f"{path}: line {line_number} has {len(fields)} fields"
            )

        rows.append([float(value) for value in fields])

    return pd.DataFrame(rows, columns=columns)


instances = []
checks = []

for path in csv_files:
    df = read_instance(path)

    instances.append({
        "activity": path.parent.name,
        "filename": path.name,
        "data": df
    })

    checks.append({
        "activity": path.parent.name,
        "filename": path.name,
        "rows": len(df),
        "sensor_columns": df.shape[1] - 1,
        "missing_values": int(df.isna().sum().sum())
    })

checks = pd.DataFrame(checks)

print("Loaded instances:", len(instances))
print("Total missing values:", checks["missing_values"].sum())
print("\nRow counts:")
print(checks["rows"].value_counts().sort_index())

display(checks.loc[checks["rows"] != 480])
display(instances[0]["data"].head())

Loaded instances: 88
Total missing values: 0

Row counts:
rows
479     1
480    87
Name: count, dtype: int64


,activity,filename,rows,sensor_columns,missing_values
56,sitting,dataset8.csv,479,6,0


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0.0,39.25,0.43,22.75,0.43,33.75,1.3
1,250.0,39.25,0.43,23.00,0.00,33.00,0.0
2,500.0,39.25,0.43,23.25,0.43,33.00,0.0
3,750.0,39.50,0.50,23.00,0.71,33.00,0.0
4,1000.0,39.50,0.50,24.00,0.00,33.00,0.0


All 88 instances have been loaded. Comma and whitespace delimiters have been handled, and trailing delimiters have been removed. No missing values were found in the loaded records. The file sitting/dataset8.csv contains 479 observations, while the other files contain 480. All available observations have been retained for feature extraction.

### (b) Training and Test Sets

In [4]:
train_instances = []
test_instances = []
split_records = []

for instance in instances:
    activity = instance["activity"]
    filename = instance["filename"]

    dataset_id = int(Path(filename).stem.replace("dataset", ""))

    test_limit = 2 if activity in ("bending1", "bending2") else 3
    is_test = dataset_id <= test_limit

    if is_test:
        test_instances.append(instance)
    else:
        train_instances.append(instance)

    split_records.append({
        "activity": activity,
        "filename": filename,
        "dataset_id": dataset_id,
        "split": "test" if is_test else "train"
    })

split_table = pd.DataFrame(split_records)

print("Training instances:", len(train_instances))
print("Test instances:", len(test_instances))

split_summary = pd.crosstab(
    split_table["activity"], split_table["split"]
).reindex(columns=["train", "test"])

display(split_summary)

Training instances: 69
Test instances: 19


split,train,test
activity,,
bending1,5,2
bending2,4,2
cycling,12,3
lying,12,3
sitting,12,3
standing,12,3
walking,12,3


### (c)(i) Common Time-Domain Features

Common time-domain features include:

- Mean and median: show the center of the values.
- Minimum and maximum: show the lowest and highest values.
- Variance and standard deviation: show how spread out the values are.
- Quartiles and interquartile range: show how the values are divided and how spread out the middle half is.
- Skewness and kurtosis: describe the shape of the distribution and its tails.
- Root mean square and energy: measure the size of the signal.
- Autocorrelation: shows how values at different times are related.

Reference: [tsfresh feature documentation](https://tsfresh.readthedocs.io/en/latest/text/list_of_features.html)

### (c)(ii) Extract Time-Domain Features

I extract seven features from each of the six sensor time series:
minimum, maximum, mean, median, standard deviation, first quartile,
and third quartile.

Each instance has 42 features. I use the sample standard deviation
(ddof=1). I keep the original feature values without scaling.
The time column is not used as a sensor feature.

In [5]:
def extract_features(df):
    result = {}

    for i, column in enumerate(columns[1:], start=1):
        values = df[column]

        result[f"min{i}"] = values.min()
        result[f"max{i}"] = values.max()
        result[f"mean{i}"] = values.mean()
        result[f"median{i}"] = values.median()
        result[f"std{i}"] = values.std(ddof=1)
        result[f"q1_{i}"] = values.quantile(0.25)
        result[f"q3_{i}"] = values.quantile(0.75)

    return result


feature_rows = [
    extract_features(instance["data"])
    for instance in instances
]

features = pd.DataFrame(feature_rows)
features.index = pd.RangeIndex(
    start=1, stop=len(features) + 1, name="Instance"
)

metadata = split_table[["activity", "filename", "split"]].copy()
metadata.index = features.index

train_features = features.loc[metadata["split"] == "train"]
test_features = features.loc[metadata["split"] == "test"]

print("Feature table shape:", features.shape)
print("Training feature shape:", train_features.shape)
print("Test feature shape:", test_features.shape)
print("Missing feature values:", features.isna().sum().sum())

display(features)

Feature table shape: (88, 42)
Training feature shape: (69, 42)
Test feature shape: (19, 42)
Missing feature values: 0


,min1,max1,mean1,median1,std1,q1_1,q3_1,min2,max2,mean2,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
2,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
3,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
4,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
5,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,4.223792,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
85,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,4.115750,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
86,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,4.396958,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200


### (c)(iii) Standard Deviations and Bootstrap Confidence Intervals

I calculate the sample standard deviation of each feature across
all 88 instances.

I use 10,000 bootstrap samples. Each sample contains 88 instances
drawn with replacement. I calculate each feature's standard deviation
for every sample.

The 5th and 95th percentiles give the 90% bootstrap confidence interval.
I use a fixed random seed so the results can be repeated.

In [6]:
import numpy as np

values = features.to_numpy()
n_instances, n_features = values.shape

feature_std = values.std(axis=0, ddof=1)

n_bootstrap = 10000
rng = np.random.default_rng(42)
bootstrap_std = np.empty((n_bootstrap, n_features))

for b in range(n_bootstrap):
    indices = rng.integers(0, n_instances, size=n_instances)
    sample = values[indices]
    bootstrap_std[b] = sample.std(axis=0, ddof=1)

ci_lower = np.percentile(bootstrap_std, 5, axis=0)
ci_upper = np.percentile(bootstrap_std, 95, axis=0)

std_summary = pd.DataFrame({
    "standard_deviation": feature_std,
    "CI_90_lower": ci_lower,
    "CI_90_upper": ci_upper
}, index=features.columns)

std_summary.index.name = "feature"

print("Number of features:", len(std_summary))
print("Bootstrap samples:", n_bootstrap)

with pd.option_context("display.max_rows", 50):
    display(std_summary.round(4))

Number of features: 42
Bootstrap samples: 10000


,standard_deviation,CI_90_lower,CI_90_upper
feature,,,
min1,9.5700,8.2591,10.7762
max1,4.3944,3.3243,5.2928
mean1,5.3357,4.6952,5.8836
median1,5.4401,4.7909,6.0076
std1,1.7722,1.5701,1.9549
q1_1,6.1536,5.5646,6.6560
q3_1,5.1389,4.3111,5.8451
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6254,5.4020


The table shows each feature's standard deviation and its 90%
bootstrap confidence interval. The features min2 and min4 have
zero standard deviation. They have the same value across all
instances, so they cannot help separate the activities on their own.

### (c)(iv) Select Three Time-Domain Features

I choose mean, standard deviation, and maximum.

- Mean shows the average signal level.
- Standard deviation shows how much the signal changes around its mean.
- Maximum shows the highest signal value.

These features describe different parts of a signal. They may help
separate activities with different signal levels and movement patterns.
I choose them based on what they measure.

In [7]:
selected_types = ["mean", "std", "max"]

selected_columns = [
    f"{feature_type}{i}"
    for i in range(1, 7)
    for feature_type in selected_types
]

selected_features = features[selected_columns].copy()

print("Selected feature types:", selected_types)
print("Selected feature table shape:", selected_features.shape)

display(selected_features.head())

Selected feature types: ['mean', 'std', 'max']
Selected feature table shape: (88, 18)


feature,mean1,std1,max1,mean2,std2,max2,mean3,std3,max3,mean4,std4,max4,mean5,std5,max5,mean6,std6,max6
Instance,,,,,,,,,,,,,,,,,,
1,40.624792,1.476967,45.00,0.358604,0.322605,1.30,19.040937,4.462952,29.50,0.832542,0.965659,7.23,34.311292,2.188449,38.25,0.570583,0.582915,1.92
2,42.812812,1.435550,45.67,0.372437,0.289158,1.22,20.096979,3.893737,29.50,0.876438,1.047259,5.76,33.024583,1.995255,38.50,0.571083,0.601010,3.11
3,43.954500,1.558835,47.40,0.426250,0.338690,1.70,22.122354,3.030943,29.75,0.497312,0.550657,4.44,35.588458,1.999604,38.50,0.493292,0.513506,1.79
4,42.179812,3.670666,47.75,0.696042,0.630860,3.00,22.183625,3.810469,30.00,0.989917,0.953730,5.15,33.493917,3.849448,38.67,0.613521,0.524317,2.18
5,41.678063,2.243490,45.75,0.535979,0.405469,2.83,19.006563,4.087107,28.25,0.841875,0.928801,6.42,29.857083,2.411026,37.50,0.383292,0.389164,1.79


## 2. ISLR 3.7.4

### (a) Linear relationship: training RSS

The cubic model has a lower or equal training RSS.

It includes all the terms in the linear model. Setting the
coefficients of X² and X³ to zero gives the linear model.
With least squares, adding these terms cannot increase training RSS.

### (b) Linear relationship: test RSS

I expect the linear model to have a lower test RSS.

The true relationship is linear, so the extra terms are not needed.
The cubic model may fit noise in the training data. This adds
variance and can hurt its predictions on new data.

This is an expected result, not a guarantee for every test set.

### (c) Nonlinear relationship: training RSS

The cubic model still has a lower or equal training RSS.

It includes the linear model as a special case. This remains true
even when the true relationship is nonlinear.

### (d) Nonlinear relationship: test RSS

There is not enough information to choose one model.

If the cubic model captures the true pattern better, it may have
a lower test RSS. If the relationship is close to linear, the
extra terms may add variance with little benefit.

The result depends on the true pattern and the amount of noise.

Reference: James et al., An Introduction to Statistical Learning
with Applications in R, 2nd edition, Section 3.7, Exercise 4.